In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName('pyspark-pilot')
    .master('local[*]')
    .getOrCreate()
)

In [17]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

schema = StructType([
    StructField('employee_id', StringType(), False),
    StructField('department_id', StringType(), False),
    StructField('name', StringType(), False),
    StructField('age', IntegerType(), True),
    StructField('gender', StringType(), True),
    StructField('salary', DoubleType(), True),
    StructField('hire_date', DateType(), True),
    StructField('tax', DoubleType(), True),
    StructField('one', StringType(), True),
    StructField('two', StringType(), True)
])
    

df = spark.read.schema(schema).csv('/home/spark/data/employees_tax', header=True).drop('one', 'two')

In [18]:
df.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- tax: double (nullable = true)



In [23]:
from pyspark.sql.functions import when, col

df.withColumn('short_gender', 
              when(col('gender') == 'Male', 'M')
              .when(col('gender') == 'Female', 'F')
              .otherwise(None)
             ).show()

+-----------+-------------+-------------+---+------+-------+----------+-------+------------+
|employee_id|department_id|         name|age|gender| salary| hire_date|    tax|short_gender|
+-----------+-------------+-------------+---+------+-------+----------+-------+------------+
|        015|          106|  Michael Lee| 37|  Male|63000.0|2014-09-30|12600.0|           M|
|        016|          107|  Kelly Zhang| 30|Female|49000.0|2018-04-01| 9800.0|           F|
|        007|          101|James Johnson| 42|  Male|70000.0|2012-03-15|14000.0|           M|
|        008|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|10200.0|           F|
|        011|          104|   David Park| 38|  Male|65000.0|2015-11-01|13000.0|           M|
|        012|          105|   Susan Chen| 31|Female|54000.0|2017-02-15|10800.0|           F|
|        019|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|12400.0|           M|
|        020|          102|    Grace Kim| 32|Female|53000.0|2018-11-01

In [25]:
from pyspark.sql.functions import to_date

df.withColumn('hire_date', to_date('hire_date', 'dd-mm-yyyy')).show()

+-----------+-------------+-------------+---+------+-------+----------+-------+----------+
|employee_id|department_id|         name|age|gender| salary| hire_date|    tax|  new_date|
+-----------+-------------+-------------+---+------+-------+----------+-------+----------+
|        015|          106|  Michael Lee| 37|  Male|63000.0|2014-09-30|12600.0|2014-09-30|
|        016|          107|  Kelly Zhang| 30|Female|49000.0|2018-04-01| 9800.0|2018-04-01|
|        007|          101|James Johnson| 42|  Male|70000.0|2012-03-15|14000.0|2012-03-15|
|        008|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|10200.0|2019-10-01|
|        011|          104|   David Park| 38|  Male|65000.0|2015-11-01|13000.0|2015-11-01|
|        012|          105|   Susan Chen| 31|Female|54000.0|2017-02-15|10800.0|2017-02-15|
|        019|          103|  Steven Chen| 36|  Male|62000.0|2015-08-01|12400.0|2015-08-01|
|        020|          102|    Grace Kim| 32|Female|53000.0|2018-11-01|10600.0|2018-11-01|

In [29]:
from pyspark.sql.functions import current_date, current_timestamp

df.withColumn('current_date', current_date()).withColumn('current_timestamp', current_timestamp()).show(truncate=False)

+-----------+-------------+-------------+---+------+-------+----------+-------+------------+--------------------------+
|employee_id|department_id|name         |age|gender|salary |hire_date |tax    |current_date|current_timestamp         |
+-----------+-------------+-------------+---+------+-------+----------+-------+------------+--------------------------+
|015        |106          |Michael Lee  |37 |Male  |63000.0|2014-09-30|12600.0|2026-11-14  |2026-11-14 02:13:24.345724|
|016        |107          |Kelly Zhang  |30 |Female|49000.0|2018-04-01|9800.0 |2026-11-14  |2026-11-14 02:13:24.345724|
|007        |101          |James Johnson|42 |Male  |70000.0|2012-03-15|14000.0|2026-11-14  |2026-11-14 02:13:24.345724|
|008        |102          |Kate Kim     |29 |Female|51000.0|2019-10-01|10200.0|2026-11-14  |2026-11-14 02:13:24.345724|
|011        |104          |David Park   |38 |Male  |65000.0|2015-11-01|13000.0|2026-11-14  |2026-11-14 02:13:24.345724|
|012        |105          |Susan Chen   

In [33]:
from pyspark.sql.functions import date_format

df.withColumn('hire_date', date_format(col('hire_date'), 'dd/MM/yyyy')).show()

+-----------+-------------+-------------+---+------+-------+----------+-------+
|employee_id|department_id|         name|age|gender| salary| hire_date|    tax|
+-----------+-------------+-------------+---+------+-------+----------+-------+
|        015|          106|  Michael Lee| 37|  Male|63000.0|30/09/2014|12600.0|
|        016|          107|  Kelly Zhang| 30|Female|49000.0|01/04/2018| 9800.0|
|        007|          101|James Johnson| 42|  Male|70000.0|15/03/2012|14000.0|
|        008|          102|     Kate Kim| 29|Female|51000.0|01/10/2019|10200.0|
|        011|          104|   David Park| 38|  Male|65000.0|01/11/2015|13000.0|
|        012|          105|   Susan Chen| 31|Female|54000.0|15/02/2017|10800.0|
|        019|          103|  Steven Chen| 36|  Male|62000.0|01/08/2015|12400.0|
|        020|          102|    Grace Kim| 32|Female|53000.0|01/11/2018|10600.0|
|        003|          102|    Bob Brown| 35|  Male|55000.0|01/05/2014|11000.0|
|        004|          102|    Alice Lee